In [146]:
import pandas as pd

dataset = pd.read_csv("../dataset/explored_raw_data.csv")

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  object 
 2   model         548438 non-null  object 
 3   trim          548186 non-null  object 
 4   body          545642 non-null  object 
 5   transmission  493485 non-null  object 
 6   vin           558833 non-null  object 
 7   state         558837 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  object 
 11  interior      558088 non-null  object 
 12  seller        558837 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558799 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 68.2+ MB


In [147]:
dataset.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,2014-12-16 12:30:00+08:00
1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,2014-12-16 12:30:00+08:00
2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,2015-01-15 04:30:00+08:00
3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,2015-01-29 04:30:00+08:00
4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,2014-12-18 12:30:00+08:00


In [148]:
# Task 1 - deletion of negligible NULL rows

dataset.dropna(subset = ["vin", "odometer", "color", "interior", "mmr", "sellingprice", "saledate"], inplace = True)


In [149]:
# Task 2 - Dropping rows where there is no basic information of Vehicle

make_null = dataset[dataset["make"].isna()].index

dataset.drop(make_null, inplace = True)


In [150]:
# Task 3 - Dropping duplicated rows of 'vin'



dataset.drop_duplicates(subset = ["vin"], inplace = True)

dataset["vin"].duplicated().sum()


np.int64(0)

In [151]:
dataset.isna().sum()

year                0
make                0
model              93
trim              345
body             2875
transmission    62562
vin                 0
state               0
condition       11551
odometer            0
color               0
interior            0
seller              0
mmr                 0
sellingprice        0
saledate            0
dtype: int64

In [152]:
dataset.groupby(["year", "make", "model", "trim", "body"])["transmission"].nunique().value_counts()

transmission
1    10419
2     3193
0      193
Name: count, dtype: int64

In [ ]:
#Task 4 - Dropping Null rows of 'model'

dataset.dropna(subset = ["model"], inplace = True)

dataset.isna().sum()

year                0
make                0
model               0
trim              345
body             2875
transmission    62557
vin                 0
state               0
condition       11551
odometer            0
color               0
interior            0
seller              0
mmr                 0
sellingprice        0
saledate            0
dtype: int64

In [ ]:
# Task 5 - Replacing null values from 'trim' to 'unknown'

dataset["trim"].fillna("unknown", inplace = True)

dataset.isna().sum()

C:\Users\prana\AppData\Local\Temp\ipykernel_6412\521296212.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataset["trim"].fillna("unknown", inplace = True)


year                0
make                0
model               0
trim                0
body             2875
transmission    62557
vin                 0
state               0
condition       11551
odometer            0
color               0
interior            0
seller              0
mmr                 0
sellingprice        0
saledate            0
dtype: int64

In [ ]:
# Task 6 - Creating group to identify where ["year", "make", "model", "trim"] uniquely indentifies 'body'

body_map = (dataset.dropna(subset = ["body"])
            .groupby(["year", "make", "model", "trim"])["body"]
            .agg(lambda x: x.iloc[0] if x.nunique() == 1 else None))

body_map

year  make       model            trim              
1990  cadillac   deville          base                        sedan
      chevrolet  c/k 1500 series  454ss                 regular cab
                 corvette         base                    hatchback
      honda      accord           ex                          sedan
                                  lx                          sedan
                                                           ...     
2015  volvo      xc60             t5 drive-e premier            suv
                                  t5 premier                    suv
                                  t6                            suv
                 xc70             3.2                         wagon
                                  t6                          wagon
Name: body, Length: 12089, dtype: object

In [ ]:
# Task 7 - Applying value to NULL values of 'body' which is uniquely determined by ["year", "make", "model", "trim"]

mask = dataset["body"].isna()

dataset.loc[mask, "body"] = (dataset.loc[mask, ["year", "make", "model", "trim"]]
                                         .apply(lambda row: body_map.get(tuple(row)), axis = 1))


dataset.isna().sum()

year                0
make                0
model               0
trim                0
body             2825
transmission    62557
vin                 0
state               0
condition       11551
odometer            0
color               0
interior            0
seller              0
mmr                 0
sellingprice        0
saledate            0
dtype: int64

In [ ]:
(dataset
.dropna(subset = ["condition", "year", "make", "model", "body", "trim"])
.groupby(["year", "make", "model", "body", "trim"])["condition"]
.median())

year  make       model            body         trim              
1990  cadillac   deville          sedan        base                   2.0
      chevrolet  c/k 1500 series  regular cab  454ss                  3.5
                 corvette         hatchback    base                   2.0
      honda      accord           sedan        ex                     1.0
                                               lx                     1.5
                                                                     ... 
2015  volvo      xc60             suv          t5 drive-e premier    45.5
                                               t5 premier            44.0
                                               t6                    41.5
                 xc70             wagon        3.2                   44.0
                                               t6                    46.0
Name: condition, Length: 13733, dtype: float64

In [ ]:
# Task 8 - Creating map for 'condition' where the median is depended on ["year", "make", "model"]

condition_map = (dataset.dropna(subset = ["condition", "year", "make", "model"])
                 .groupby(["year", "make", "model"])["condition"]
                 .agg("median"))

condition_map

year  make       model   
1984  chevrolet  corvette     2.0
1985  chevrolet  corvette     4.0
1986  chevrolet  corvette     4.0
1987  mercedes   300e         2.0
1989  chevrolet  corvette     3.5
                             ... 
2015  volvo      s60         42.0
                 s80         42.0
                 v60         42.5
                 xc60        45.0
                 xc70        46.0
Name: condition, Length: 5307, dtype: float64

In [ ]:
# Task 9 - Filling NULL conditions with the median

mask = dataset["condition"].isna()

dataset.loc[mask, "condition"] = (dataset.loc[mask, ["year", "make", "model"]]
                                  .apply(lambda row: condition_map.get(tuple(row)), axis = 1))

dataset.isna().sum()
# Still 82 NULL values left

year                0
make                0
model               0
trim                0
body             2825
transmission    62557
vin                 0
state               0
condition          82
odometer            0
color               0
interior            0
seller              0
mmr                 0
sellingprice        0
saledate            0
dtype: int64

In [ ]:
# Task 10 - Removing 82 NULL rows of condition (due to its size)

dataset.dropna(subset = ["condition"], inplace = True)

dataset.isna().sum()

year                0
make                0
model               0
trim                0
body             2768
transmission    62532
vin                 0
state               0
condition           0
odometer            0
color               0
interior            0
seller              0
mmr                 0
sellingprice        0
saledate            0
dtype: int64

In [162]:
# Task 11 - Save the Cleaned Dataset in new csv file

dataset.to_csv("../dataset/cleaned_dataset.csv")